# JOIN em SQL

JOIN é utilizado para combinar dados de duas ou mais tabelas relacionadas.

Em bancos relacionais, é comum que os dados sejam separados em diferentes tabelas. Por exemplo:

- `clientes` armazena os clientes;
- `pedidos` armazena os pedidos;
- `produtos` armazena os produtos;
- `itens_pedido` relaciona os produtos aos pedidos.

O JOIN permite consultar essas informações em conjunto.

Os exemplos deste material utilizam a estrutura criada nos materiais anteriores.

## Preparando os dados

Considere as seguintes tabelas:

```sql
CREATE TABLE clientes (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    email VARCHAR(150) UNIQUE NOT NULL
);

CREATE TABLE pedidos (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    cliente_id INTEGER NOT NULL,
    data_pedido DATE NOT NULL,
    valor NUMERIC(10, 2) NOT NULL,
    FOREIGN KEY (cliente_id) REFERENCES clientes(id)
);

CREATE TABLE produtos (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    preco NUMERIC(10, 2) NOT NULL
);

CREATE TABLE itens_pedido (
    pedido_id INTEGER NOT NULL,
    produto_id INTEGER NOT NULL,
    quantidade INTEGER NOT NULL,
    PRIMARY KEY (pedido_id, produto_id),
    FOREIGN KEY (pedido_id) REFERENCES pedidos(id),
    FOREIGN KEY (produto_id) REFERENCES produtos(id)
);
```

Podemos inserir alguns dados para realizar os exemplos:

```sql
INSERT INTO clientes (nome, email) VALUES
    ('Ana', 'ana@email.com'),
    ('Bruno', 'bruno@email.com'),
    ('Carlos', 'carlos@email.com'),
    ('Daniela', 'daniela@email.com');

INSERT INTO pedidos (cliente_id, data_pedido, valor) VALUES
    (1, '2026-09-01', 150.00),
    (1, '2026-09-05', 80.00),
    (2, '2026-09-03', 220.00);

INSERT INTO produtos (nome, preco) VALUES
    ('Teclado', 120.00),
    ('Mouse', 80.00),
    ('Monitor', 900.00);

INSERT INTO itens_pedido (pedido_id, produto_id, quantidade) VALUES
    (1, 1, 1),
    (1, 2, 1),
    (2, 2, 1),
    (3, 3, 1);
```

## O conceito de JOIN

A estrutura básica é:

```sql
SELECT colunas
FROM tabela_a
JOIN tabela_b
    ON tabela_a.coluna = tabela_b.coluna;
```

A cláusula `ON` define como os registros das tabelas estão relacionados.

Por exemplo, `pedidos.cliente_id` referencia `clientes.id`:

```sql
SELECT
    clientes.nome,
    pedidos.id,
    pedidos.valor
FROM clientes
JOIN pedidos
    ON clientes.id = pedidos.cliente_id;
```

O resultado combina informações das duas tabelas.

## INNER JOIN

O `INNER JOIN` retorna somente os registros que possuem correspondência nas duas tabelas.

```sql
SELECT
    clientes.nome,
    pedidos.id AS pedido,
    pedidos.valor
FROM clientes
INNER JOIN pedidos
    ON clientes.id = pedidos.cliente_id;
```

Também é possível escrever simplesmente `JOIN`:

```sql
SELECT
    clientes.nome,
    pedidos.id AS pedido,
    pedidos.valor
FROM clientes
JOIN pedidos
    ON clientes.id = pedidos.cliente_id;
```

Nesse caso, `JOIN` significa `INNER JOIN`.

No exemplo, clientes que ainda não possuem pedidos não aparecem no resultado.

## Utilizando aliases

Aliases deixam consultas com JOIN mais curtas e legíveis.

```sql
SELECT
    c.nome,
    p.id AS pedido,
    p.valor
FROM clientes AS c
JOIN pedidos AS p
    ON c.id = p.cliente_id;
```

O `AS` pode ser omitido:

```sql
SELECT
    c.nome,
    p.id AS pedido,
    p.valor
FROM clientes c
JOIN pedidos p
    ON c.id = p.cliente_id;
```

Aliases são especialmente úteis quando uma consulta envolve várias tabelas.

## LEFT JOIN

O `LEFT JOIN` retorna todos os registros da tabela que está à esquerda do JOIN, mesmo quando não existe correspondência na tabela da direita.

```sql
SELECT
    c.nome,
    p.id AS pedido,
    p.valor
FROM clientes c
LEFT JOIN pedidos p
    ON c.id = p.cliente_id;
```

Nesse exemplo, todos os clientes aparecem.

Se um cliente não possuir pedidos, as colunas de `pedidos` terão valor `NULL`.

Isso é útil quando queremos encontrar registros que não possuem relacionamento:

```sql
SELECT
    c.id,
    c.nome
FROM clientes c
LEFT JOIN pedidos p
    ON c.id = p.cliente_id
WHERE p.id IS NULL;
```

Essa consulta retorna clientes que ainda não fizeram nenhum pedido.

## RIGHT JOIN

O `RIGHT JOIN` é semelhante ao `LEFT JOIN`, mas preserva todos os registros da tabela que está à direita.

```sql
SELECT
    c.nome,
    p.id AS pedido,
    p.valor
FROM clientes c
RIGHT JOIN pedidos p
    ON c.id = p.cliente_id;
```

Nesse exemplo, todos os pedidos aparecem, mesmo que algum pedido não possua um cliente correspondente.

Em projetos reais, `LEFT JOIN` costuma ser mais utilizado porque geralmente é mais fácil organizar a consulta colocando a tabela que queremos preservar à esquerda.

## FULL OUTER JOIN

O `FULL OUTER JOIN` retorna todos os registros das duas tabelas.

Quando existe correspondência, os dados são combinados.

Quando não existe correspondência, as colunas da outra tabela ficam com `NULL`.

```sql
SELECT
    c.nome,
    p.id AS pedido,
    p.valor
FROM clientes c
FULL OUTER JOIN pedidos p
    ON c.id = p.cliente_id;
```

O `FULL OUTER JOIN` é útil quando queremos identificar tanto registros sem correspondência de um lado quanto do outro.

## CROSS JOIN

O `CROSS JOIN` combina cada registro de uma tabela com todos os registros da outra.

```sql
SELECT
    c.nome,
    pr.nome AS produto
FROM clientes c
CROSS JOIN produtos pr;
```

Se existirem 4 clientes e 3 produtos, o resultado terá:

```text
4 × 3 = 12 registros
```

O `CROSS JOIN` deve ser utilizado com cuidado, pois pode produzir uma quantidade muito grande de registros.

## JOIN entre várias tabelas

É possível utilizar vários JOINs na mesma consulta.

Por exemplo, para descobrir quais produtos foram vendidos em cada pedido:

```sql
SELECT
    p.id AS pedido,
    c.nome AS cliente,
    pr.nome AS produto,
    ip.quantidade
FROM pedidos p
JOIN clientes c
    ON c.id = p.cliente_id
JOIN itens_pedido ip
    ON ip.pedido_id = p.id
JOIN produtos pr
    ON pr.id = ip.produto_id;
```

A consulta percorre os relacionamentos:

```text
clientes
    ↓
pedidos
    ↓
itens_pedido
    ↓
produtos
```

Esse é um dos usos mais comuns de JOIN em aplicações reais.

## JOIN com condições adicionais

A condição do relacionamento fica normalmente no `ON`, enquanto filtros sobre o resultado podem ficar no `WHERE`.

```sql
SELECT
    c.nome,
    p.id AS pedido,
    p.valor
FROM clientes c
JOIN pedidos p
    ON c.id = p.cliente_id
WHERE p.valor >= 100;
```

Também podemos adicionar condições ao próprio relacionamento:

```sql
SELECT
    c.nome,
    p.id AS pedido,
    p.valor
FROM clientes c
LEFT JOIN pedidos p
    ON c.id = p.cliente_id
    AND p.valor >= 100;
```

Essas duas consultas podem produzir resultados diferentes.

No `LEFT JOIN`, colocar uma condição no `ON` permite manter os registros da tabela da esquerda mesmo quando nenhum registro da direita atende à condição.

## Cuidado com WHERE e LEFT JOIN

Considere:

```sql
SELECT
    c.nome,
    p.id AS pedido,
    p.valor
FROM clientes c
LEFT JOIN pedidos p
    ON c.id = p.cliente_id
WHERE p.valor >= 100;
```

O `WHERE` elimina as linhas em que `p.valor` é `NULL`.

Na prática, isso pode fazer a consulta se comportar como um `INNER JOIN` para essa condição.

Se a intenção for preservar todos os clientes e apenas relacionar pedidos com valor mínimo de 100:

```sql
SELECT
    c.nome,
    p.id AS pedido,
    p.valor
FROM clientes c
LEFT JOIN pedidos p
    ON c.id = p.cliente_id
    AND p.valor >= 100;
```

Essa diferença é importante quando trabalhamos com `LEFT JOIN`.

## JOIN com a mesma tabela

Uma tabela também pode ser relacionada com ela mesma. Isso é chamado de `SELF JOIN`.

Considere uma tabela de funcionários:

```sql
CREATE TABLE funcionarios (
    id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    gerente_id INTEGER,
    FOREIGN KEY (gerente_id) REFERENCES funcionarios(id)
);
```

Podemos descobrir quem é o gerente de cada funcionário:

```sql
SELECT
    f.nome AS funcionario,
    g.nome AS gerente
FROM funcionarios f
LEFT JOIN funcionarios g
    ON f.gerente_id = g.id;
```

Os aliases `f` e `g` são necessários para diferenciar as duas referências à mesma tabela.

## JOIN e nomes de colunas

Quando duas tabelas possuem colunas com o mesmo nome, é importante informar de qual tabela a coluna pertence.

Por exemplo:

```sql
SELECT
    c.id,
    c.nome,
    p.id,
    p.valor
FROM clientes c
JOIN pedidos p
    ON c.id = p.cliente_id;
```

É possível utilizar aliases para deixar o resultado mais claro:

```sql
SELECT
    c.id AS cliente_id,
    c.nome AS cliente,
    p.id AS pedido_id,
    p.valor
FROM clientes c
JOIN pedidos p
    ON c.id = p.cliente_id;
```

Qualificar as colunas também evita ambiguidades em consultas maiores.

## JOIN usando USING

Quando as duas tabelas possuem uma coluna com exatamente o mesmo nome e ela é utilizada no relacionamento, o PostgreSQL permite utilizar `USING`.

Por exemplo:

```sql
SELECT *
FROM tabela_a
JOIN tabela_b
    USING (id);
```

Nesse caso, `id` precisa existir nas duas tabelas.

Para as tabelas do nosso exemplo, o relacionamento utiliza nomes diferentes:

```text
clientes.id
pedidos.cliente_id
```

Por isso, utilizamos `ON`:

```sql
SELECT *
FROM clientes c
JOIN pedidos p
    ON c.id = p.cliente_id;
```

`ON` é mais geral e deixa explícita a condição do relacionamento.

## Ordem lógica da consulta

Uma consulta com JOIN pode ser escrita nesta estrutura:

```sql
SELECT colunas
FROM tabela_principal
JOIN outra_tabela
    ON condição_do_relacionamento
WHERE filtros
ORDER BY colunas;
```

Um exemplo:

```sql
SELECT
    c.nome,
    p.id AS pedido,
    p.valor
FROM clientes c
JOIN pedidos p
    ON c.id = p.cliente_id
WHERE p.valor >= 100
ORDER BY p.valor DESC;
```

Uma forma simples de entender a consulta é:

- `FROM` define a tabela inicial;
- `JOIN` adiciona outra tabela;
- `ON` define como as tabelas se relacionam;
- `WHERE` filtra os registros;
- `SELECT` define as colunas exibidas;
- `ORDER BY` define a ordenação.